In [ ]:
# --- 1. Imports and Dataset Loading ---
import os
import sys
import subprocess

try:
    import kagglehub
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', 'kagglehub'])
    import kagglehub

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from IPython.display import display

pd.options.display.float_format = '{:.2f}'.format

# Download and load the NASA Kepler cumulative dataset.
path = kagglehub.dataset_download('nasa/kepler-exoplanet-search-results')
full_path = os.path.join(path, 'cumulative.csv')
df = pd.read_csv(full_path)

print('Dataset loaded successfully.')
print(f'Dataset shape: {df.shape[0]} rows, {df.shape[1]} columns')
display(df.head())

In [ ]:
# --- 2. First Data Inspection ---

print('Shape:')
print(df.shape)

print('\nColumn names:')
print(df.columns.tolist())

print('\nDataFrame information:')
df.info()

print('\nFirst rows:')
display(df.head())

In [ ]:
# --- 3. Dataset Dictionary Guided Cleaning ---
# This is the single cleaning step used by the whole notebook.
# Metadata and leakage columns are removed from the predictive dataframe.
# False-positive diagnostic flags are preserved separately for descriptive analysis only.

RANDOM_STATE = 42
TARGET_COLUMN = 'koi_disposition'
POSITIVE_LABEL = 'CONFIRMED'
NEGATIVE_LABEL = 'FALSE POSITIVE'
CANDIDATE_LABEL = 'CANDIDATE'

# Diagnostic false-positive flags from the data dictionary.
# They are post-analysis conclusions, so using them as features would leak the answer.
fp_flag_cols = [
    'koi_fpflag_nt',
    'koi_fpflag_ss',
    'koi_fpflag_co',
    'koi_fpflag_ec'
]

# Metadata columns: useful for cataloging, not for learning physical patterns.
metadata_cols = [
    'rowid',
    'kepid',
    'kepoi_name',
    'koi_tce_delivname',
    'koi_tce_plnt_num'
]

# Leakage columns: they encode naming, preliminary classification, or NASA probability scores.
leakage_cols = [
    'kepler_name',
    'koi_pdisposition',
    'koi_score'
]

# Structurally empty columns found in the missing-value audit.
empty_cols = ['koi_teq_err1', 'koi_teq_err2']

columns_to_drop = metadata_cols + leakage_cols + fp_flag_cols

# Keep flags separately before dropping them from the cleaned predictive dataframe.
df_false_positive_flags = df.loc[
    df[TARGET_COLUMN].eq(NEGATIVE_LABEL),
    [TARGET_COLUMN, *fp_flag_cols]
].copy()

# df_clean is the unified dataframe for EDA and modeling.
# It still contains koi_disposition because the target is needed later.
df_clean = df.drop(columns=[c for c in columns_to_drop if c in df.columns])

print(f'Columns before cleaning: {df.shape[1]}')
print(f'Columns after metadata/leakage removal: {df_clean.shape[1]}')
print(f'False-positive flag rows preserved: {df_false_positive_flags.shape[0]}')
print('Dropped columns:')
print(columns_to_drop)

In [ ]:
# --- 4. Target Distribution with Categorical Tools ---
print('Target counts:')
display(df_clean[TARGET_COLUMN].value_counts(dropna=False).to_frame('count'))

print('Target proportions:')
display(df_clean[TARGET_COLUMN].value_counts(normalize=True, dropna=False).to_frame('proportion'))

plt.figure(figsize=(7, 4))
sns.countplot(data=df_clean, x=TARGET_COLUMN, hue=TARGET_COLUMN, palette='Set1', legend=False)
plt.title('KOI Disposition Counts')
plt.xlabel('Disposition')
plt.ylabel('Count')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

print('Project decision: CANDIDATE observations are excluded from supervised training because they are unresolved labels.')

In [ ]:
# --- 5. Missing Values Audit with G01 Tools ---
missing_counts = df_clean.isnull().sum()
missing_percentage = (missing_counts / len(df_clean)) * 100

missing_df = pd.DataFrame({
    'Missing Values': missing_counts,
    'Percentage (%)': missing_percentage
})

missing_df = missing_df[missing_df['Missing Values'] > 0].sort_values('Missing Values', ascending=False)

print('Missing-value audit after metadata/leakage removal:')
display(missing_df)

print('Important modeling rule: missing values are not imputed here. Imputation will be fitted inside Pipelines after train/test split.')

In [ ]:
# --- 6. Numerical Descriptive Statistics with G01 Tools ---
key_num_vars = [
    'koi_period',
    'koi_duration',
    'koi_depth',
    'koi_prad',
    'koi_teq',
    'koi_insol',
    'koi_model_snr',
    'koi_steff',
    'koi_slogg',
    'koi_srad',
    'koi_kepmag'
]
key_num_vars = [c for c in key_num_vars if c in df_clean.columns]

print('Descriptive statistics for key physical variables:')
display(df_clean[key_num_vars].describe())

summary_rows = []
for col in key_num_vars:
    x = df_clean[col]
    summary_rows.append({
        'feature': col,
        'mean': x.mean(),
        'median': x.median(),
        'std': x.std(),
        'iqr': x.quantile(0.75) - x.quantile(0.25),
        'min': x.min(),
        'max': x.max()
    })

summary_table = pd.DataFrame(summary_rows)
display(summary_table)

In [ ]:
# --- 7. Necessary Visual EDA with G01 Plots ---
# Histograms: check skewness/shape for physically important variables.
# Speed choice: no KDE, because KDE can be slow on skewed variables.
# Log scale choice: use logarithmic bins, not only a log x-axis, so the bars are meaningful.
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, feature in zip(axes, ['koi_prad', 'koi_period', 'koi_depth']):
    if feature in df_clean.columns:
        plot_data = df_clean[df_clean[feature] > 0]
        log_bins = np.logspace(np.log10(plot_data[feature].min()), np.log10(plot_data[feature].max()), 30)
        sns.histplot(data=plot_data, x=feature, bins=log_bins, alpha=0.4, edgecolor='white', ax=ax)
        ax.set_xscale('log')
        ax.set_title(f'Distribution of {feature} (log bins)')
plt.tight_layout()
plt.show()

# Boxplots: compare key variables across the target categories.
# The y-axis is log-scaled to make class differences visible despite extreme values.
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, feature in zip(axes, ['koi_prad', 'koi_period', 'koi_model_snr']):
    if feature in df_clean.columns:
        plot_data = df_clean[df_clean[feature] > 0]
        sns.boxplot(data=plot_data, x=TARGET_COLUMN, y=feature, hue=TARGET_COLUMN, palette='Set1', legend=False, ax=ax)
        ax.set_yscale('log')
        ax.set_title(f'{feature} by disposition (log scale)')
        ax.tick_params(axis='x', rotation=15)
plt.tight_layout()
plt.show()

# Scatterplot: inspect the relationship between orbital period and planetary radius.
# Speed/readability choice: use a reproducible sample if the dataset is large.
# Both axes are log-scaled because period and radius cover very different orders of magnitude.
if {'koi_period', 'koi_prad'}.issubset(df_clean.columns):
    scatter_data = df_clean[(df_clean['koi_period'] > 0) & (df_clean['koi_prad'] > 0)]
    scatter_data = scatter_data.sample(n=min(2000, len(scatter_data)), random_state=RANDOM_STATE)
    plt.figure(figsize=(7, 5))
    sns.scatterplot(data=scatter_data, x='koi_period', y='koi_prad', hue=TARGET_COLUMN, alpha=0.6, palette='Set1', s=25)
    plt.xscale('log')
    plt.yscale('log')
    plt.title('Period vs Planet Radius by Disposition (log-log scale)')
    plt.tight_layout()
    plt.show()

# Correlation: simple linear relationships among selected numerical variables.
corr_vars = [c for c in ['koi_period', 'koi_duration', 'koi_depth', 'koi_prad', 'koi_teq', 'koi_insol', 'koi_model_snr', 'koi_steff', 'koi_srad'] if c in df_clean.columns]
plt.figure(figsize=(9, 6))
sns.heatmap(df_clean[corr_vars].corr(numeric_only=True), annot=False, cmap='Blues', vmin=-1, vmax=1)
plt.title('Correlation Matrix of Key Numerical Variables')
plt.tight_layout()
plt.show()

print('EDA conclusion: false positives show visible differences in physical/transit variables, while CANDIDATE is not a final supervised label.')

In [ ]:
# --- 8. Binary Modeling Dataset and Correct Split Order ---
# This cell begins the supervised-learning workflow.
# The order is: remove CANDIDATE -> define X/y -> train/test split -> impute inside Pipelines.

from sklearn.model_selection import train_test_split

# Keep only final labels for the binary supervised task.
df_binary = df_clean[df_clean[TARGET_COLUMN].isin([POSITIVE_LABEL, NEGATIVE_LABEL])].copy()

# Binary target: confirmed exoplanet = 1, false positive = 0.
y = df_binary[TARGET_COLUMN].map({NEGATIVE_LABEL: 0, POSITIVE_LABEL: 1})

# Build X from the cleaned dataframe.
# No imputation, scaling, fitting, or model training has happened before this split.
X = df_binary.drop(columns=[TARGET_COLUMN, *[c for c in empty_cols if c in df_binary.columns]])
X = X.select_dtypes(include=['float64', 'int64'])

# Safeguards against leakage and target mistakes.
assert set(df_binary[TARGET_COLUMN].unique()) == {POSITIVE_LABEL, NEGATIVE_LABEL}
assert CANDIDATE_LABEL not in set(df_binary[TARGET_COLUMN].unique())
assert y.notna().all()
assert not any(c in X.columns for c in columns_to_drop)
assert not any(c in X.columns for c in empty_cols)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)

print('Binary modeling dataset prepared.')
print(f'Rows after removing CANDIDATE: {df_binary.shape[0]}')
print(f'Predictive features: {X.shape[1]}')
print(f'Training set: {X_train.shape[0]} rows')
print(f'Test set: {X_test.shape[0]} rows')
print('\nTarget distribution:')
print(y.map({0: NEGATIVE_LABEL, 1: POSITIVE_LABEL}).value_counts())
print('\nExcluded columns still in X:')
print([c for c in columns_to_drop + empty_cols if c in X.columns])